# Predictor 1: Speech Envelope

This notebook builds only the broadband speech envelope predictor for the Alice comprehension acoustic tracking analysis.

It reads stimulus WAV files from the BIDS data directory and writes this analysis-specific predictor into the separate Alice Comprehension workspace.

## Output convention

Input:

```text
/Users/yanyuwoo/Data/bids/stimuli/*.wav
```

Output:

```text
/Users/yanyuwoo/Data/Alice Comprehension/predictors/speech_envelope/
/Users/yanyuwoo/Data/Alice Comprehension/qc/speech_envelope_manifest.csv
```

Each output `.npz` contains:

- `data`: shape `(time, 1)`, sampled at 100 Hz
- `metadata`: JSON string with source file and construction parameters

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import hilbert, butter, filtfilt, resample_poly

BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')
STIMULI_DIR = BIDS_ROOT / 'stimuli'

ANALYSIS_ROOT = Path('/Users/yanyuwoo/Data/Alice Comprehension')
PREDICTOR_DIR = ANALYSIS_ROOT / 'predictors' / 'speech_envelope'
QC_DIR = ANALYSIS_ROOT / 'qc'

PREDICTOR_FS = 100
ENVELOPE_LOWPASS_HZ = 30.0

PREDICTOR_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

## Helper functions

In [3]:
def read_wav_mono(path: Path):
    fs, audio = wavfile.read(path)
    audio = np.asarray(audio, dtype=np.float64)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    max_abs = np.max(np.abs(audio))
    if max_abs > 0:
        audio = audio / max_abs
    return fs, audio


def zscore_column(x):
    x = np.asarray(x, dtype=np.float64)
    if not np.isfinite(x).all():
        raise ValueError('Cannot z-score data with non-finite values')
    sd = x.std()
    if sd == 0:
        raise ValueError('Cannot z-score constant predictor')
    return ((x - x.mean()) / sd).reshape(-1, 1)


def lowpass(x, fs, cutoff_hz):
    nyquist = fs / 2
    cutoff_hz = min(cutoff_hz, nyquist * 0.95)
    b, a = butter(4, cutoff_hz / nyquist, btype='low')
    return filtfilt(b, a, x)


def resample_to_predictor_fs(x, source_fs, target_fs=PREDICTOR_FS):
    if source_fs == target_fs:
        return x
    gcd = np.gcd(int(source_fs), int(target_fs))
    up = int(target_fs // gcd)
    down = int(source_fs // gcd)
    return resample_poly(x, up, down)


def speech_envelope(audio, fs):
    envelope = np.abs(hilbert(audio))
    envelope = lowpass(envelope, fs, ENVELOPE_LOWPASS_HZ)
    envelope = resample_to_predictor_fs(envelope, fs, PREDICTOR_FS)
    return zscore_column(envelope)


def save_predictor(path: Path, data: np.ndarray, metadata: dict):
    np.savez_compressed(
        path,
        data=data.astype(np.float32),
        metadata=json.dumps(metadata),
    )

## Build speech envelope predictors

Run this cell when you are ready to generate the first predictor.

In [4]:
wav_paths = sorted(STIMULI_DIR.glob('*.wav'), key=lambda p: int(p.stem))
manifest_rows = []

for wav_path in wav_paths:
    stimulus_id = int(wav_path.stem)
    fs, audio = read_wav_mono(wav_path)
    predictor = speech_envelope(audio, fs)

    out_path = PREDICTOR_DIR / f'stim-{stimulus_id:02d}_speech_envelope_fs-{PREDICTOR_FS}.npz'
    metadata = {
        'stimulus_id': stimulus_id,
        'predictor_name': 'speech_envelope',
        'source_wav': str(wav_path),
        'source_fs': fs,
        'predictor_fs': PREDICTOR_FS,
        'envelope_lowpass_hz': ENVELOPE_LOWPASS_HZ,
        'source_duration_sec': len(audio) / fs,
        'n_samples': int(predictor.shape[0]),
        'n_features': int(predictor.shape[1]),
    }
    save_predictor(out_path, predictor, metadata)

    manifest_rows.append({
        **metadata,
        'path': str(out_path),
        'predictor_duration_sec': predictor.shape[0] / PREDICTOR_FS,
        'predictor_mean': float(predictor.mean()),
        'predictor_std': float(predictor.std()),
    })

manifest = pd.DataFrame(manifest_rows).sort_values('stimulus_id')
manifest_path = QC_DIR / 'speech_envelope_manifest.csv'
manifest.to_csv(manifest_path, index=False)
manifest

,stimulus_id,predictor_name,source_wav,source_fs,predictor_fs,envelope_lowpass_hz,source_duration_sec,n_samples,n_features,path,predictor_duration_sec,predictor_mean,predictor_std
0,1,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/1.wav,44100,100,30.0,57.540612,5755,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,57.55,-1.086495e-16,1.0
1,2,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/2.wav,44100,100,30.0,60.845193,6085,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,60.85,1.167696e-17,1.0
2,3,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/3.wav,44100,100,30.0,63.259433,6326,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,63.26,-1.123210e-17,1.0
3,4,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/4.wav,44100,100,30.0,69.988571,6999,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,69.99,-4.466907e-17,1.0
4,5,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/5.wav,44100,100,30.0,66.272540,6628,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,66.28,-3.430502e-17,1.0
5,6,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/6.wav,44100,100,30.0,63.777551,6378,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,63.78,1.203177e-16,1.0
6,7,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/7.wav,44100,100,30.0,62.896848,6290,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,62.90,-6.325977e-17,1.0
7,8,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/8.wav,44100,100,30.0,57.310612,5732,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,57.32,-8.925170e-17,1.0
8,9,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/9.wav,44100,100,30.0,57.226145,5723,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,57.23,1.291219e-16,1.0
9,10,speech_envelope,/Users/yanyuwoo/Data/bids/stimuli/10.wav,44100,100,30.0,61.269660,6127,1,/Users/yanyuwoo/Data/Alice Comprehension/predi...,61.27,2.087444e-17,1.0
